In [ ]:
from pathlib import Path
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, energy_distance, wasserstein_distance
from scipy.spatial.distance import jensenshannon

ROOT = Path.cwd()
RESULTS_ROOT = ROOT / "number_extracted"
GT_ROOT = ROOT / "ground_truth_values"

print("ROOT:", ROOT)
print("RESULTS_ROOT exists:", RESULTS_ROOT.exists())
print("GT_ROOT exists:", GT_ROOT.exists())

In [ ]:
# Load full ground-truth values by uid and keep first-100 view for task dictionary
ground_truth_full_by_uid = {}
ground_truth_by_uid = {}
gt_files = sorted(GT_ROOT.rglob("ground_truth_values.json"))

for gt_file in gt_files:
    with gt_file.open("r", encoding="utf-8") as f:
        rows = json.load(f)

    for row in rows:
        uid = row.get("uid")
        values = row.get("ground_truth_values", [])
        if uid is None:
            continue

        values = [float(v) for v in values if isinstance(v, (int, float)) and np.isfinite(v)]
        ground_truth_full_by_uid[uid] = values
        ground_truth_by_uid[uid] = values[:100]

num_uids = len(ground_truth_full_by_uid)
num_uids_with_300 = sum(1 for v in ground_truth_full_by_uid.values() if len(v) >= 300)
print(f"Loaded ground truth for {num_uids:,} UIDs from {len(gt_files)} files.")
print(f"UIDs with at least 300 GT samples: {num_uids_with_300:,}")

In [ ]:
# Build task dictionary keyed by uid
# Each UID stores task metadata, ground truth values, and per-model extracted numbers.
tasks = {}
result_files = sorted(RESULTS_ROOT.rglob("all_results.json"))

for result_file in result_files:
    with result_file.open("r", encoding="utf-8") as f:
        rows = json.load(f)

    for row in rows:
        uid = row.get("uid")
        if uid is None:
            continue

        if uid not in tasks:
            tasks[uid] = {
                "uid": uid,
                "id": row.get("id"),
                "category": row.get("category"),
                "subcategory": row.get("subcategory"),
                "prompt_title": row.get("prompt_title"),
                "ground_truth_values": ground_truth_by_uid.get(uid, []),
                "model_values": defaultdict(list),
            }

        model_name = row.get("model_used")
        extracted = row.get("extracted_number")

        if model_name is None:
            continue

        if isinstance(extracted, (int, float)) and np.isfinite(extracted):
            tasks[uid]["model_values"][model_name].append(float(extracted))

# Convert defaultdicts to plain dicts for easier inspection and export.
for uid in tasks:
    tasks[uid]["model_values"] = dict(tasks[uid]["model_values"])

print(f"Loaded {len(result_files):,} result files.")
print(f"Built task dictionary for {len(tasks):,} UIDs.")

In [ ]:
from scipy.stats import gaussian_kde
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance, energy_distance


def ks_score(a, b):
    """Calculate Kolmogorov-Smirnov test."""
    stat, p_value = ks_2samp(a, b)
    return {
        "ks_statistic": float(stat),
        "ks_p_value": float(p_value),
    }


def _w1_sorted_equal(x_sorted, y_sorted):
    """Wasserstein-1 between two equal-size 1D samples, both pre-sorted."""
    return np.mean(np.abs(x_sorted - y_sorted))


def _w1_sorted(x, y):
    """Wasserstein-1 for 1D samples of possibly unequal size. Sorts internally."""
    x = np.sort(x)
    y = np.sort(y)
    if x.size == y.size:
        return np.mean(np.abs(x - y))
    # Fallback: scipy handles unequal sizes via the CDF-integral form
    return wasserstein_distance(x, y)


def distance_distribution_scores(a, b, n_resamples=999, rng=None):
    """Debiased Wasserstein-1 and energy distance with a shared permutation null."""
    rng = np.random.default_rng(rng)
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    n_a = a.size
    n_b = b.size
    if n_a < 2 or n_b < 2:
        return {k: float("nan") for k in (
            "wasserstein_debiased", "wasserstein_z",
            "energy_debiased", "energy_z",
        )}

    pooled = np.concatenate([a, b])
    n_total = pooled.size
    equal_sizes = (n_a == n_b)

    # Observed statistics
    if equal_sizes:
        w_obs = _w1_sorted_equal(np.sort(a), np.sort(b))
    else:
        w_obs = _w1_sorted(a, b)
    e_obs = float(energy_distance(a, b))

    # Shared permutation null
    w_null = np.empty(n_resamples)
    e_null = np.empty(n_resamples)
    for i in range(n_resamples):
        perm = rng.permutation(n_total)
        x = pooled[perm[:n_a]]
        y = pooled[perm[n_a:]]
        if equal_sizes:
            w_null[i] = _w1_sorted_equal(np.sort(x), np.sort(y))
        else:
            w_null[i] = _w1_sorted(x, y)
        e_null[i] = energy_distance(x, y)

    def _summarize(obs, null):
        mean = null.mean()
        std = null.std()
        debiased = float(obs - mean)
        if std < 1e-12:
            z = float("nan")
        else:
            z = float((obs - mean) / std)
        return debiased, z

    w_debiased, w_z = _summarize(w_obs, w_null)
    e_debiased, e_z = _summarize(e_obs, e_null)

    return {
        "wasserstein_debiased": w_debiased,
        "wasserstein_z": w_z,
        "energy_debiased": e_debiased,
        "energy_z": e_z,
    }


def js_divergence_score(a, b, grid_size=512, pad=0.1):
    """Jensen-Shannon divergence via KDE on a shared grid."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if a.size < 2 or b.size < 2:
        return np.nan
    if np.allclose(a.min(), a.max()) and np.allclose(b.min(), b.max()):
        return 0.0 if np.isclose(a[0], b[0]) else np.nan

    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    span = hi - lo
    lo -= pad * span
    hi += pad * span
    grid = np.linspace(lo, hi, grid_size)

    try:
        p = gaussian_kde(a)(grid)
        q = gaussian_kde(b)(grid)
    except (np.linalg.LinAlgError, ValueError):
        return np.nan

    p_sum, q_sum = p.sum(), q.sum()
    if p_sum == 0 or q_sum == 0:
        return np.nan
    p /= p_sum
    q /= q_sum

    js_distance = jensenshannon(p, q)
    return float(js_distance ** 2)


RANDOM_MODEL_NAME = "random/last-100-gt"


def compute_metrics_records(gt_start, gt_end, include_random_baseline=True):
    records = []

    for uid, task in tasks.items():
        gt_full = ground_truth_full_by_uid.get(uid, [])
        gt_slice = gt_full[gt_start:gt_end]
        if len(gt_slice) == 0:
            continue

        gt_arr = np.asarray(gt_slice, dtype=float)

        for model_name, model_vals in task.get("model_values", {}).items():
            if len(model_vals) == 0:
                continue

            pred_arr = np.asarray(model_vals, dtype=float)
            ks = ks_score(gt_arr, pred_arr)
            dist_scores = distance_distribution_scores(gt_arr, pred_arr)

            rec = {
                "uid": uid,
                "prompt_title": task.get("prompt_title"),
                "model": model_name,
                "ks_statistic": ks["ks_statistic"],
                "ks_p_value": ks["ks_p_value"],
                "js_divergence": js_divergence_score(gt_arr, pred_arr),
            }
            rec.update(dist_scores)
            records.append(rec)

        if include_random_baseline and len(gt_full) >= 100:
            random_pred = np.asarray(gt_full[-100:], dtype=float)
            ks = ks_score(gt_arr, random_pred)
            dist_scores = distance_distribution_scores(gt_arr, random_pred)

            rec = {
                "uid": uid,
                "prompt_title": task.get("prompt_title"),
                "model": RANDOM_MODEL_NAME,
                "ks_statistic": ks["ks_statistic"],
                "ks_p_value": ks["ks_p_value"],
                "js_divergence": js_divergence_score(gt_arr, random_pred),
            }
            rec.update(dist_scores)
            records.append(rec)

    return records

In [ ]:
metrics_df = pd.DataFrame(compute_metrics_records(0, 1000, include_random_baseline=True))

In [ ]:
metrics_df_avg = metrics_df[['ks_statistic', 'ks_p_value', 'js_divergence', 'wasserstein_debiased', 'wasserstein_z', 'energy_debiased', 'energy_z', 'model']].groupby('model').mean()
metrics_df_avg = metrics_df_avg.sort_values('ks_p_value', ascending=False)
metrics_df_avg